## Configure local inputs

Set the following environment variables to your files or folders before launching Jupyter: `MOUSEGPT_OUTPUT_DIR`, `MOUSEGPT_SESSION_DIR`. Paths may include spaces. Output variables select destinations; input variables select existing session data. Review the analysis parameters in the cells before running on a new experiment.


In [ ]:
import os
required_paths = ['MOUSEGPT_OUTPUT_DIR', 'MOUSEGPT_SESSION_DIR']
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    raise ValueError("Set these environment variables before running: " + ", ".join(missing_paths))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from glob import glob
import os

In [ ]:
import os
def merge_frame_write(frame_list, out_vid):
    h = [cv2.hconcat(frame_list[i*4:i*4+4]) for i in range(2)]
    v = cv2.vconcat(h)
    
    out_vid.write(cv2.resize(v, (1920, 1080)))
    

data_2D = np.load('multi_mouse_4.npy')

indices = np.where((data_2D==0).all(axis=(1,2,3)))[0]

folder_path = os.environ['MOUSEGPT_SESSION_DIR']

caps = [cv2.VideoCapture(vid) for vid in sorted(glob(f'{folder_path}/data*.mp4'))]

is_Zero = False
Zero_num = 0
Zero_list = []

out_folder = os.environ['MOUSEGPT_OUTPUT_DIR']

os.makedirs(out_folder, exist_ok=True)

fourc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
fps = 30
height = 1080
width = 1920
out_vid = None

In [ ]:
indices


In [ ]:
for index in tqdm(range(len(indices)-1)):
    if indices[index] + 1 == indices[index + 1]:
        if is_Zero:
            frame_list = []
            for cap in caps:
                _, frame = cap.read()
                frame_list.append(frame)
            merge_frame_write(frame_list, out_vid)
        else:
            out_vid = cv2.VideoWriter(f'{out_folder}/{indices[index]}.mp4', fourc, fps, (width, height))
            frame_list = []
            for cap in caps:
                cap.set(cv2.CAP_PROP_POS_FRAMES, indices[index])
                _, frame = cap.read()
                frame_list.append(frame)
            merge_frame_write(frame_list, out_vid)
            is_Zero = True
    else:
        if is_Zero:
            out_vid.release()
            is_Zero = False